Notebook to test on real data

In [1]:
import numpy as np
import pandas as pd
import duckdb as duckdb

import psutil, os

from datkit.cleaning import clean_dataframe
from datkit.binning import bin_column

In [2]:
p = psutil.Process(os.getpid())

psutil.virtual_memory().available / 1024 / 1024    # headroom on the machine

6557.08984375

In [11]:
p.memory_info().rss    / 1024 / 1024                 # what this process holds right now

1824.33203125

In [9]:
p.memory_info().peak_wset  / 1024 / 1024             # Windows only: high-water mark

1917.13671875

Import table

In [2]:
# don't use dtype=str as that creates legacy objects that pythons stores strings in
# pyarrow is a better option for string storage and manipulation (columnar storage)
df_title_basics = pd.read_csv(
    "d:/dev/data-analysis-toolkit/devdata/title.basics.tsv.gz", sep="\t", dtype_backend="pyarrow"
)

df_title_akas = pd.read_csv(
    "d:/dev/data-analysis-toolkit/devdata/title.akas.tsv.gz", sep="\t", dtype_backend="pyarrow"
)


In [3]:

df_title_ratings = pd.read_csv(
    "d:/dev/data-analysis-toolkit/devdata/title.ratings.tsv.gz", sep="\t", dtype_backend="pyarrow"
)

## clean table

In [4]:
df_title_basics.sample(n=5, random_state=1)

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres
2615756,tt13020394,tvEpisode,Episode #1.544,Episode #1.544,0,\N,\N,\N,Drama
8913370,tt37283892,tvSeries,Lost Boy,Lost Boy,0,\N,\N,\N,Thriller
11115785,tt6285120,tvEpisode,Episode #8.5,Episode #8.5,0,2016,\N,55,"Game-Show,Romance"
5583584,tt21887000,tvEpisode,Episode #1.1808,Episode #1.1808,0,\N,\N,\N,\N
9032730,tt37782317,tvEpisode,08-18-2025,08-18-2025,0,2025,\N,\N,Talk-Show


In [5]:
df_title_ratings.sample(n=5, random_state=1)

,tconst,averageRating,numVotes
1544457,tt6464184,6.8,290
26936,tt0044583,5.4,66
922261,tt1986692,8.1,17
1705361,tt9769820,8.2,56
462316,tt0953069,7.8,16


In [ ]:
df_title_basics, title_basics_cleaning_report = clean_dataframe(df_title_basics, True)

In [28]:
df_title_ratings, title_ratingscleaning_report = clean_dataframe(df_title_ratings, True)

In [29]:
df_title_basics.sample(n=5, random_state=1)

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres
2615756,tt13020394,tvEpisode,Episode #1.544,Episode #1.544,0,<NA>,<NA>,<NA>,Drama
8913370,tt37283892,tvSeries,Lost Boy,Lost Boy,0,<NA>,<NA>,<NA>,Thriller
11115785,tt6285120,tvEpisode,Episode #8.5,Episode #8.5,0,2016,<NA>,55,"Game-Show,Romance"
5583584,tt21887000,tvEpisode,Episode #1.1808,Episode #1.1808,0,<NA>,<NA>,<NA>,<NA>
9032730,tt37782317,tvEpisode,08-18-2025,08-18-2025,0,2025,<NA>,<NA>,Talk-Show


In [30]:
title_basics_cleaning_report

,column,dtype_before,dtype_after,converted,bytes_before,bytes_after,bytes_saved,mb_before,mb_after,mb_saved,nulls_before,nulls_after,nulls_created
0,tconst,string[pyarrow],string[pyarrow],False,173132806,174728871,-1596065,165.11,166.63,-1.52,0,0,0
1,titleType,string[pyarrow],string[pyarrow],False,156599336,158195401,-1596065,149.34,150.87,-1.52,0,0,0
2,primaryTitle,string[pyarrow],string[pyarrow],False,307956611,308013920,-57309,293.69,293.74,-0.05,25,54,29
3,originalTitle,string[pyarrow],string[pyarrow],False,307996903,308054212,-57309,293.73,293.78,-0.05,25,54,29
4,isAdult,int64[pyarrow],int64[pyarrow],False,102148128,102148128,0,97.42,97.42,0.00,0,0,0
5,startYear,string[pyarrow],int64[pyarrow],True,99184912,103744193,-4559281,94.59,98.94,-4.35,0,1481608,1481608
6,endYear,string[pyarrow],int64[pyarrow],True,76933517,103744193,-26810676,73.37,98.94,-25.57,0,12607294,12607294
7,runtimeMinutes,string[pyarrow],string[pyarrow],False,76424418,61633033,14791385,72.88,58.78,14.11,0,8193725,8193725
8,genres,string[pyarrow],string[pyarrow],False,191478276,191795418,-317142,182.61,182.91,-0.30,32,541877,541845


In [ ]:
df_title_ratings.sample(n=5, random_state=1)

,tconst,averageRating,numVotes
1544457,tt6464184,6.8,290
26936,tt0044583,5.4,66
922261,tt1986692,8.1,17
1705361,tt9769820,8.2,56
462316,tt0953069,7.8,16


In [ ]:
title_ratingscleaning_report

,column,dtype_before,dtype_after,converted,bytes_before,bytes_after,bytes_saved,mb_before,mb_after,mb_saved,nulls_before,nulls_after,nulls_created
0,tconst,string[pyarrow],string[pyarrow],False,22763052,22976955,-213903,21.71,21.91,-0.2,0,0,0
1,averageRating,double[pyarrow],double[pyarrow],False,13689792,13689792,0,13.06,13.06,0.0,0,0,0
2,numVotes,int64[pyarrow],int64[pyarrow],False,13689792,13689792,0,13.06,13.06,0.0,0,0,0


In [ ]:
df_title_ratings["tconst"].nunique()

1711224

In [ ]:
df_title_basics["tconst"].nunique()

12768516

###Binning

In [19]:
df = bin_column(df_title_ratings["averageRating"])

In [20]:
df

,bin,bin_min,bin_max,width,count,pct
0,"[-inf, 2.1)",1.000,2.100,1.100,7756,0.45
1,"[2.1, 2.495)",2.100,2.495,0.395,4983,0.29
2,"[2.495, 2.89)",2.495,2.890,0.395,8519,0.50
3,"[2.89, 3.285)",2.890,3.285,0.395,10522,0.61
4,"[3.285, 3.68)",3.285,3.680,0.395,15523,0.91
5,"[3.68, 4.075)",3.680,4.075,0.395,20519,1.20
6,"[4.075, 4.47)",4.075,4.470,0.395,28671,1.68
7,"[4.47, 4.865)",4.470,4.865,0.395,40055,2.34
8,"[4.865, 5.26)",4.865,5.260,0.395,55934,3.27
9,"[5.26, 5.655)",5.260,5.655,0.395,78014,4.56


In [21]:
bin_column(df_title_basics["runtimeMinutes"])

,bin,bin_min,bin_max,width,count,pct
0,30,<NA>,<NA>,1.0,509754,3.99
1,60,<NA>,<NA>,1.0,373320,2.92
2,22,<NA>,<NA>,1.0,227350,1.78
3,45,<NA>,<NA>,1.0,200171,1.57
4,15,<NA>,<NA>,1.0,126733,0.99
5,25,<NA>,<NA>,1.0,114750,0.90
6,10,<NA>,<NA>,1.0,93704,0.73
7,44,<NA>,<NA>,1.0,93378,0.73
8,23,<NA>,<NA>,1.0,87429,0.68
9,5,<NA>,<NA>,1.0,82807,0.65


## Explore

In [36]:
# Merge using duckdb

df_title_basics = duckdb.sql(
    """ 
    select 
        a.*, b.averageRating, b.numVotes
    from 
        df_title_basics a
        left join
        df_title_ratings b
        on
            a.tconst = b.tconst
    """
    ).df()

In [21]:
# Merge using pandas merge

df_title_basics = df_title_basics.merge(df_title_ratings[['averageRating', 'numVotes','tconst']], on='tconst', how='left')

In [46]:
df_title_basics.sample(n=5, random_state=1)

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres,averageRating,numVotes,tooOld,hasRating
276826,tt0475351,short,Opera Baby,Opera Baby,0,2005.0,NaN,3.0,"Comedy,Short",NaN,NaN,False,False
849425,tt0927940,tvEpisode,Ui-ui-ui oder Die Verwandlung der Energie,Ui-ui-ui oder Die Verwandlung der Energie,0,1983.0,NaN,NaN,Drama,NaN,NaN,True,False
504499,tt0235631,movie,Wet Lust: 21 Strippers,Nureta yokujô: Tokudashi 21-nin,0,1974.0,NaN,77.0,"Comedy,Drama",6.8,53.0,True,True
601054,tt0606283,tvEpisode,Episode dated 3 February 2001,Episode dated 3 February 2001,0,2001.0,NaN,45.0,Talk-Show,NaN,NaN,False,False
980221,tt0997102,movie,SuperWizja,SuperWizja,0,1993.0,NaN,83.0,"Mystery,Sci-Fi",5.5,24.0,True,True


In [44]:
# Create flags for conditions

df_title_basics["tooOld"] = df_title_basics["startYear"] < 2000
df_title_basics["hasRating"] = df_title_basics["averageRating"].notna()


In [45]:
df_title_basics.groupby(['titleType']).agg(
    all_records=('tconst', 'count'),
    distinct_keys=('tconst', 'nunique'),
    Numtooold=('tooOld', 'sum'),
    Numwrating=('hasRating', 'sum'),
)

,all_records,distinct_keys,Numtooold,Numwrating
titleType,,,,
movie,226294,226294,190326,152153
short,139792,139792,104722,51770
tvEpisode,467210,467210,244857,182416
tvMiniSeries,6149,6149,4177,4549
tvMovie,45395,45395,30782,26315
tvSeries,44349,44349,27746,27699
tvShort,2004,2004,1239,904
tvSpecial,6755,6755,3349,3977
video,56702,56702,23204,24705


In [18]:
duckdb.sql(
    """ 
    select 
        titleType, avg(isAdult) average_Adult
    from 
        df_title_basics
    group by
        titleType
    """
    ).show()

┌──────────────┬───────────────────────┐
│  titleType   │     average_Adult     │
│   varchar    │        double         │
├──────────────┼───────────────────────┤
│ tvMovie      │    0.0005577352104008 │
│ short        │  0.002432588902860001 │
│ tvSpecial    │ 0.0006018456600240739 │
│ videoGame    │  0.049801393241382066 │
│ video        │   0.34985629795734574 │
│ tvShort      │ 0.0002714440825190011 │
│ movie        │  0.012029918090235627 │
│ tvPilot      │                   0.0 │
│ tvSeries     │  0.010280220501378133 │
│ tvEpisode    │   0.03503159613674168 │
│ tvMiniSeries │ 0.0066859265373503925 │
├──────────────┴───────────────────────┤
│ 11 rows                    2 columns │
└──────────────────────────────────────┘



In [ ]:
df_title_basics_clean.shape()

In [ ]:
def sample_records(df: pd.DataFrame, n=10000, seed=1) -> pd.DataFrame:
    """Return a sample of records from the DataFrame."""
    if len(df) > n:
        return df.sample(n=n, random_state=seed)
    else:
        return df

In [ ]:
df_clean_sample = sample_records(df_clean)

In [ ]:
df_sample_nonunique = df_clean_sample.nunique()

In [ ]:
df_sample_nonunique.iloc[2]

np.int64(9835)

In [ ]:
df_sample_nonunique["isAdult"]

np.int64(2)

In [ ]:
cleaning_report[cleaning_report["column"] == "isAdult"]["dtype_after"].values[0]

'int64[pyarrow]'

for col in df_clean_sample.columns:
    if df_sample_nonunique[col] > 100:
        if "string" in cleaning_report[cleaning_report["column"] == col]["dtype_after"].values[0]:
            pass
        else:
            df_clean_sample[col] = pd.cut(df_clean_sample[col], bins=5)
    elif df_sample_nonunique[col] > 10:
    else:

def binning(s:pd.Series) -> pd.Series:
    

In [ ]:
df_clean_sample["runtimeMinutes_bins"] = pd.cut(df_clean_sample["runtimeMinutes"], bins=10)

In [ ]:
df_clean_sample.groupby("runtimeMinutes_bins", observed=False)["runtimeMinutes"].agg(["count", "min", "max", "mean", "median", "std"])

,count,min,max,mean,median,std
runtimeMinutes_bins,,,,,,
"(-0.427, 143.7]",8549,1,143,74.047023,83.0,30.93211
"(143.7, 286.4]",178,144,282,187.747191,177.5,35.500905
"(286.4, 429.1]",33,288,420,335.606061,320.0,41.296897
"(429.1, 571.8]",3,476,540,512.0,520.0,32.741411
"(571.8, 714.5]",1,624,624,624.0,624.0,<NA>
"(714.5, 857.2]",1,763,763,763.0,763.0,<NA>
"(857.2, 999.9]",0,<NA>,<NA>,<NA>,<NA>,<NA>
"(999.9, 1142.6]",0,<NA>,<NA>,<NA>,<NA>,<NA>
"(1142.6, 1285.3]",0,<NA>,<NA>,<NA>,<NA>,<NA>


In [ ]:
series_nona = df_clean_sample["runtimeMinutes"].dropna()
low, high = np.percentile(series_nona, [1,99]) # does not accept NA values
print(low, high)

6.0 210.0


In [ ]:
arrasy_sample = np.array([1,2,2,3,4,4,4,5,5,6,6,7,7,8])
np.unique(arrasy_sample)

array([1, 2, 3, 4, 5, 6, 7, 8])

In [ ]:
binresult = bin_column(df_clean_sample["titleType"], tail_min_span=0.3)
binresult

TypeError: bin_categorical() got an unexpected keyword argument 'tail_min_span'

In [ ]:
binresult

,region,left,right,count,pct
0,low_tail,1.0,3.0,32,0.37
1,core,3.0,29.2,1035,11.81
2,core,29.2,55.4,883,10.07
3,core,55.4,81.6,2156,24.60
4,core,81.6,107.8,3731,42.56
5,core,107.8,134.0,639,7.29
6,core,134.0,160.2,124,1.41
7,core,160.2,186.4,55,0.63
8,core,186.4,212.6,26,0.30
9,core,212.6,238.8,22,0.25
